# Notebook 03: Model Training, Hyperparameter Tuning & Comparison
**Author:** Mark Eliezer M. Villola | ProTech Devices Philippines | AIM Capstone 2025

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0,'..')
RANDOM_STATE=42; np.random.seed(RANDOM_STATE)
os.makedirs('../models',exist_ok=True); os.makedirs('../reports/figures',exist_ok=True)
print('Environment ready.')

## 1. Load Feature Data and Split

In [ ]:
from sklearn.model_selection import train_test_split
try:
    df = pd.read_parquet('../data/processed/02_features_output.parquet')
    print(f'Loaded: {df.shape}')
except FileNotFoundError:
    from fraud_detection_pipeline import generate_demo_data, clean_data, engineer_features, select_features
    df_raw = generate_demo_data(n_samples=40000)
    df_raw = clean_data(df_raw); df_raw = engineer_features(df_raw)
    feats = select_features(df_raw, target='isFraud', max_features=55)
    df = df_raw[[c for c in feats if c in df_raw.columns] + ['isFraud']]

TARGET = 'isFraud'
feat_cols = [c for c in df.columns if c != TARGET]
X = df[feat_cols]; y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,} | Fraud rate: {y.mean()*100:.2f}%')

# Save test set for notebook 04
test_df = X_test.copy(); test_df[TARGET] = y_test
test_df.to_parquet('../data/processed/03_test_set.parquet', index=False)

## 2. Train All Models via Pipeline

In [ ]:
from fraud_detection_pipeline import train_all_models
import pandas as pd
trained_models, all_results = train_all_models(X_train, X_test, y_train, y_test)
print('\nAll models trained.')

## 3. Results Summary

In [ ]:
results_clean = [{k:v for k,v in r.items() if k not in ('y_prob','y_pred')} for r in all_results]
results_df = pd.DataFrame(results_clean).sort_values('AUC-ROC', ascending=False)
print('MODEL COMPARISON RESULTS')
print(results_df.to_string(index=False))
results_df.to_csv('../reports/model_comparison_results.csv', index=False)
print('\nResults saved.')

## 4. Save Model Artifacts

In [ ]:
for name, model in trained_models.items():
    path = f'../models/{name}.pkl'
    joblib.dump(model, path)
    print(f'Saved: {path}')
joblib.dump(feat_cols, '../models/selected_features.pkl')
print('All artifacts saved.')